# ESC-Fault starter: diagnose electrostatic chuck faults

Simulated sensor data from 140 electrostatic chucks (the part of an etch tool that clamps and cools the wafer).
This notebook builds per-chuck drift features, trains a random forest for **task 1 (diagnose)** and **task 3
(remaining life)**, scores them on the validation split, and writes test prediction files.

- Code, full task definitions and scoring: https://github.com/forgottenl5/esc-fault-benchmark
- Also on Hugging Face: https://huggingface.co/datasets/forgottenl5/esc-fault-benchmark

The data is **simulated**. Good scores here do not prove a method works on a real tool.

In [1]:
import glob, json
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import f1_score, classification_report

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) or glob.glob(f"./**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]

wafers = {s: pd.read_parquet(find(f"wafers_{s}.parquet")) for s in ["train", "val", "test"]}
tasks = {(k, s): pd.read_csv(find(f"task{k}_{s}.csv")) for k in (1, 3) for s in ["train", "val", "test"]}
{s: w.shape for s, w in wafers.items()}

{'train': (87116, 21), 'val': (12081, 21), 'test': (25790, 21)}

## Features

Every chuck runs healthy for its first 100 wafers. Features compare the last 30 wafers with that reference,
recipe by recipe (ratios for leakage and helium, differences for temperatures, pressure and release time).
All features are **causal**: they use only wafers up to the end wafer.

In [2]:
import numpy as np
import pandas as pd

FAULTS = ["healthy", "contamination", "wear", "he_leak", "residual_charge", "heater_zone", "sensor_drift"]
REF_WAFERS = 100        # first wafers of every chuck are healthy (all onsets are at wafer 150 or later)
WINDOW = 30             # wafers summarised for the "current" state
SEV_LABEL = 0.15        # task 1: a window counts as faulty once severity >= 0.15
RUL_FRACS = [0.2, 0.4, 0.6, 0.8]

RATIO = ["i_leak_mean", "he_p_set_mean", "he_p_mean", "he_flow_mean"]
DIFF = ["ch_press_mean", "t_zone_spread", "t_wafer_mean", "t_z1_mean", "t_z2_mean", "t_z3_mean", "t_z4_mean"]
FEATURE_NAMES = ([f"r_{c}" for c in RATIO] + [f"d_{c}" for c in DIFF] +
                 ["d_dechuck_med", "stick_rate", "d_p_gap", "zone_dev_max", "flow_per_press", "wafers_since_install"])


def _per_recipe(df, cols, agg):
    return df.groupby("recipe")[cols].agg(agg)


def window_features(w, end):
    """w: per-wafer table of ONE chuck (sorted by wafer_idx). Uses wafers <= end only."""
    ref = w.iloc[:REF_WAFERS]
    cur = w.iloc[max(REF_WAFERS, end + 1 - WINDOW):end + 1]
    if len(cur) == 0:
        cur = w.iloc[end + 1 - WINDOW:end + 1]
    cols = RATIO + DIFF + ["dechuck_time_s"]
    rr, cc = _per_recipe(ref, cols, "median"), _per_recipe(cur, cols, "median")
    common = cc.index.intersection(rr.index)
    rr, cc = rr.loc[common], cc.loc[common]
    weight = cur.recipe.value_counts().reindex(common).values.astype(float)
    weight /= weight.sum()
    f = {}
    for c in RATIO:
        f[f"r_{c}"] = float(np.dot(weight, cc[c] / rr[c]))
    for c in DIFF:
        f[f"d_{c}"] = float(np.dot(weight, cc[c] - rr[c]))
    f["d_dechuck_med"] = float(np.dot(weight, cc["dechuck_time_s"] - rr["dechuck_time_s"]))
    base = ref.dechuck_time_s.median() + 2.0
    f["stick_rate"] = float((cur.dechuck_time_s > base).mean())
    f["d_p_gap"] = f["r_he_p_mean"] - f["r_he_p_set_mean"]
    f["zone_dev_max"] = max(abs(f[f"d_t_z{k}_mean"] - np.mean([f[f"d_t_z{j}_mean"] for j in range(1, 5)]))
                            for k in range(1, 5))
    f["flow_per_press"] = f["r_he_flow_mean"] / max(f["r_he_p_mean"], 1e-6)
    f["wafers_since_install"] = float(w.wafers_since_install.iloc[end])
    return [f[k] for k in FEATURE_NAMES]


def feature_matrix(wafers, rows):
    """rows: DataFrame with chuck_id, end_wafer. Returns np.array (len(rows), n_features)."""
    groups = {cid: g.sort_values("wafer_idx").reset_index(drop=True) for cid, g in wafers.groupby("chuck_id")}
    return np.array([window_features(groups[r.chuck_id], int(r.end_wafer)) for r in rows.itertuples()])

### Extended features (trends, step size, cross-signal consistency)

On top of the 21 base features: for key signals, the level now, the change over the last 30 and 60 wafers, the slope over 90 wafers and the largest 10-wafer step; plus consistency checks (does the helium setpoint respond to the zone temperature change, as it does for a real heater fault but not for a drifting sensor?). Still causal.

In [3]:
SERIES_RATIO = ["i_leak_mean", "he_flow_mean", "he_p_set_mean"]
SERIES_DIFF = ["ch_press_mean", "t_wafer_mean", "dz_mean", "dz_max", "dechuck_time_s"]
LONG = 90


def _normalised(w):
    """Per-wafer series relative to the chuck's own first-100-wafer reference, recipe by recipe."""
    ref = w.iloc[:REF_WAFERS]
    cols = list(set(RATIO + DIFF + ["dechuck_time_s"]))
    med = ref.groupby("recipe")[cols].median()
    out = pd.DataFrame(index=w.index)
    rec = w.recipe.values
    base = med.reindex(rec)
    base.index = w.index
    for c in SERIES_RATIO:
        out[c] = w[c] / base[c]
    zk = ["t_z1_mean", "t_z2_mean", "t_z3_mean", "t_z4_mean"]
    dz = np.column_stack([w[c].values - base[c].values for c in zk])
    out["dz_mean"] = dz.mean(axis=1)
    dev = dz - dz.mean(axis=1, keepdims=True)
    out["dz_max"] = dev[np.arange(len(dev)), np.abs(dev).argmax(axis=1)]
    for c in ["ch_press_mean", "t_wafer_mean", "dechuck_time_s"]:
        out[c] = w[c] - base[c]
    return out


NAMES2 = ([f"{s}_{k}" for s in SERIES_RATIO + SERIES_DIFF for k in ("now", "d30", "d60", "slope", "jump")] +
          ["dz_mean_now_x_pset", "pset_per_dz", "press_per_flow", "leak_per_flow", "wafers_since_clean"])


def window_features2(w, end, norm=None):
    base = window_features(w, end)
    if norm is None:
        norm = _normalised(w)
    lo = max(REF_WAFERS, end + 1 - LONG)
    seg = norm.iloc[lo:end + 1]
    f = []
    for c in SERIES_RATIO + SERIES_DIFF:
        v = seg[c].values
        n = len(v)
        now = np.median(v[-30:]) if n else 0.0
        p1 = np.median(v[-60:-30]) if n > 40 else now
        p2 = np.median(v[-90:-60]) if n > 70 else p1
        x = np.arange(n) - (n - 1) / 2
        slope = float((x * (v - v.mean())).sum() / max((x * x).sum(), 1e-9) * 30) if n > 10 else 0.0
        k = 10
        med = np.array([np.median(v[i:i + k]) for i in range(0, max(n - k + 1, 1), 5)]) if n >= 2 * k else np.array([now, now])
        jump = float(np.max(np.abs(np.diff(med)))) if len(med) > 1 else 0.0
        f += [float(now), float(now - p1), float(p1 - p2), slope, jump]
    g = dict(zip([f"{s}_{k}" for s in SERIES_RATIO + SERIES_DIFF for k in ("now", "d30", "d60", "slope", "jump")], f))
    dz, ps = g["dz_mean_now"], g["he_p_set_mean_now"] - 1
    f += [dz * ps, ps / (abs(dz) + 0.15),
          g["ch_press_mean_now"] / (abs(g["he_flow_mean_now"] - 1) + 0.1),
          (g["i_leak_mean_now"] - 1) / (abs(g["he_flow_mean_now"] - 1) + 0.1),
          float(w.wafers_since_clean.iloc[end])]
    return list(base) + f


FEATURE_NAMES2 = FEATURE_NAMES + NAMES2


def feature_matrix2(wafers, rows):
    groups = {cid: g.sort_values("wafer_idx").reset_index(drop=True) for cid, g in wafers.groupby("chuck_id")}
    norms = {cid: _normalised(g) for cid, g in groups.items()}
    return np.array([window_features2(groups[r.chuck_id], int(r.end_wafer), norms[r.chuck_id]) for r in rows.itertuples()])

In [4]:
X = {s: feature_matrix2(wafers[s], tasks[(1, s)]) for s in ["train", "val", "test"]}
y = {s: tasks[(1, s)].label.values for s in ["train", "val"]}
pd.DataFrame(X["train"], columns=FEATURE_NAMES2).describe().T.head(8)

,count,mean,std,min,25%,50%,75%,max
r_i_leak_mean,5880.0,1.105223,0.182790,0.758241,1.023259,1.048935,1.089474,2.102098
r_he_p_set_mean,5880.0,0.935934,0.113364,0.364299,0.924270,0.963242,0.996517,1.143221
r_he_p_mean,5880.0,0.931248,0.114693,0.364178,0.915929,0.959948,0.994835,1.142848
r_he_flow_mean,5880.0,1.163553,0.571709,0.364751,0.961143,1.019084,1.096322,4.682500
d_ch_press_mean,5880.0,0.058847,0.405546,-2.105673,-0.087238,0.026347,0.161052,1.631351
d_t_zone_spread,5880.0,0.241277,0.798639,-0.841381,-0.061349,0.010734,0.098974,5.723851
d_t_wafer_mean,5880.0,0.019085,0.394714,-0.975640,-0.192070,-0.013427,0.171724,4.128481
d_t_z1_mean,5880.0,0.000861,0.611451,-3.911167,-0.318981,0.013354,0.339223,2.405777


## Task 1: diagnose (7 classes, macro-F1)

In [5]:
clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=0, n_jobs=-1)
clf.fit(X["train"], y["train"])
pred_val = clf.predict(X["val"])
print("validation macro-F1:", round(f1_score(y["val"], pred_val, average="macro"), 3))
print(classification_report(y["val"], pred_val, digits=3))

validation macro-F1: 0.859
                 precision    recall  f1-score   support

  contamination      1.000     1.000     1.000        32
        he_leak      1.000     1.000     1.000        31
        healthy      0.910     0.962     0.935       105
    heater_zone      0.727     0.941     0.821        34
residual_charge      1.000     0.955     0.977        22
   sensor_drift      0.500     0.208     0.294        24
           wear      1.000     0.969     0.984        32

       accuracy                          0.904       280
      macro avg      0.877     0.862     0.859       280
   weighted avg      0.890     0.904     0.890       280



`sensor_drift` is the hard class: the sensor is wrong but the chuck is fine, and the train set has only 14 such chucks spread over 7 sub-types. The extended features help (test F1 0.23 to 0.43 for the random forest); try more features or a model on the raw time series.

In [6]:
pd.Series(clf.feature_importances_, index=FEATURE_NAMES2).sort_values(ascending=False).head(10)

dechuck_time_s_now    0.079223
he_flow_mean_now      0.068488
r_he_flow_mean        0.066670
zone_dev_max          0.065307
d_dechuck_med         0.063367
flow_per_press        0.060886
r_he_p_set_mean       0.053709
he_p_set_mean_now     0.042841
i_leak_mean_now       0.036405
r_he_p_mean           0.035047
dtype: float64

## Task 3: remaining life for chucks that wear out (MAE in wafers)

In [7]:
def rul_rows(labels_meta):
    rows = []
    for m in labels_meta:
        if m["run_to_failure"]:
            for e in range(m["onset_wafer"], m["n_wafers"] - 1, 10):
                rows.append((m["chuck_id"], e, m["n_wafers"] - 1 - e))
    return pd.DataFrame(rows, columns=["chuck_id", "end_wafer", "rul"])

tr = rul_rows(json.load(open(find("chucks_train.json"))))
reg = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=0, n_jobs=-1)
reg.fit(feature_matrix(wafers["train"], tr), tr.rul)
v = tasks[(3, "val")]
pred = reg.predict(feature_matrix(wafers["val"], v))
print("validation MAE (wafers):", round(np.abs(pred - v.rul).mean(), 1),
      "| guess-the-mean MAE:", round(np.abs(tr.rul.mean() - v.rul).mean(), 1))

validation MAE (wafers): 73.2 | guess-the-mean MAE: 130.6


## Test predictions

Test answers are held back. To get a test score, open an issue on GitHub with these files
(`task1: chuck_id,end_wafer,label` and `task3: chuck_id,end_wafer,rul_pred`).

In [8]:
sub1 = tasks[(1, "test")].assign(label=clf.predict(X["test"]))
sub1.to_csv("task1_submission.csv", index=False)
t3 = tasks[(3, "test")]
sub3 = t3.assign(rul_pred=reg.predict(feature_matrix(wafers["test"], t3)).round(1))
sub3.to_csv("task3_submission.csv", index=False)
sub1.label.value_counts()

label
healthy            216
heater_zone         78
wear                69
contamination       60
he_leak             51
residual_charge     47
sensor_drift        39
Name: count, dtype: int64